## Pod Configuration
A minimal configuration looks like:

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: my-nginx-server
  labels:
    app: web
spec:
  containers:
    - name: nginx
      image: nginx:1.25
      ports:
        - containerPort: 80
```

`metadata.labels` contains key value pairs associated with the pod that can be used elsewhere (like in Service) for grouping and selection. Example:
```sh
$ kubectl get pods -l app=web
```

`containerPort` declares the port on which the process inside the container is listening on. Unlike Docker's `-p` flag it does not open, bind or expose the port to the host. Thus it is mostly for documentation purposes.

Typically, pods contain one or more containers where we have a main container running the application and other sidecar container. An example configuration for that:
```yaml
apiVersion: v1
kind: Pod
metadata:
  name: my-application
spec:
  initContainers:
    - name: wait-for-db
      image: busybox:1.36
      command: ['sh', '-c', 'until nc -z db-service 5432; do sleep 2; done;']

  containers:
    - name: app
      image: myapp:v1
      volumeMounts:
        - name: shared-logs
          mountPath: /var/log/app

    - name: log-shipper
      image: fluent-bit:2.2
      volumeMounts:
        - name: shared-logs
          mountPath: /var/log/app

  volumes:
    - name: shared-logs
      emptyDir: {}
```

`initContainers` specify a container that runs to completion before app containers start. In this case *app* and *log-shipper* will not start until *wait-for-db* has completed. Here it is trying to ping the DB to check if connection can be established.

`volumeMounts` specifies a volume that can be mounted at `mountPath` inside the container. `volume` section defines the volume used here. `emptyDir` is a volume that exists for the lifetime of the pod. There are other options available which are persistent in nature.

There are configuration available that indicate the compute resource requirement:
```yaml
spec:
  containers:
    - name: api-service
      image: api-service:2.4.0      
      
      
      resources:
        requests: # Select a node that has atleast this much capacity
          cpu: "250m"
          memory: "256Mi"
        limits: # Kernel prevents the container from exceeding this limit
          cpu: "1000m"
          memory: "512Mi"
```

There is also a facility to inject environment variable inside the container:
```yaml
spec:
  containers:
    - name: api-service
      image: api-service:2.4.0

      env:
        - name: "ENVIRONMENT"
          value: "production"
```

We can also specify what happens when a container inside the pod exits using `restartPolicy`. `kubelet` deployed on each node uses the liveness probe for that:
```yaml
spec:
  # Options :
  #  Always: restart container on failure and on normal exit
  #  OnFailure: restart only if container exits non-zero
  #  Never: never restart 
  restartPolicy: Always
  terminationGracePeriodSeconds: 30
  containers:
    - name: api-service
      image: api-service:2.4.0
      # Liveness probe fails -> kubelet kills the container -> restart policy decides what happens next
      livenessProbe:
        httpGet:
          path: /healthz
          port: 8080
        initialDelaySeconds: 15 # Wait for how long after container starts to start probe
        periodSeconds: 10
      # Liveness probe fails -> removed from Service's endpoints
      readinessProbe:
        httpGet:
          path: /ready
          port: 8080
        initialDelaySeconds: 5
        periodSeconds: 5
```
So container being down is `kubelet`'s issue. Pod being down is deployment's problem. When liveness probe is missing, `kubelet` doesn't do any health checks.

There are some other advanced configurations as shown below:
```yaml
spec:
  # Security context applied at the Pod level
  securityContext:
    runAsNonRoot: true
    runAsUser: 10001
    fsGroup: 2000

  # Node targeting
  nodeSelector:
    topology.kubernetes.io/zone: us-east-1a

  # Affinity rules (e.g., avoid co-locating with other frontend pods)
  affinity:
    podAntiAffinity:
      preferredDuringSchedulingIgnoredDuringExecution:
        - weight: 100
          podAffinityTerm:
            labelSelector:
              matchExpressions:
                - key: app
                  operator: In
                  values: ["frontend"]
            topologyKey: "kubernetes.io/hostname"

  # Tolerate specific node taints (e.g., dedicated GPU nodes)
  tolerations:
    - key: "workload"
      operator: "Equal"
      value: "gpu"
      effect: "NoSchedule"
```